In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.silver;

In [0]:
CREATE OR REPLACE TABLE workspace.silver.utility_kpis AS

WITH typed AS (
    SELECT
        company,
        CAST(report_year AS INT) AS report_year,
        kpi_name,
        value_as_reported,
        TRY_CAST(REPLACE(value_as_reported, ',', '') AS DOUBLE) AS value_parsed,
        unit_as_reported,
        scope_or_note,
        source_document,
        CAST(source_page AS INT) AS source_page,
        _ingested_at
    FROM workspace.bronze.utility_kpis
),

normalized AS (
    SELECT
        *,
        CASE unit_as_reported
            WHEN 'million t CO2e'   THEN value_parsed * 1000000
            WHEN 'thousand t CO2e'  THEN value_parsed * 1000
            WHEN 't CO2e'           THEN value_parsed
            WHEN 'million MWh'      THEN value_parsed * 1000000
            WHEN 'MWh'              THEN value_parsed
            WHEN 'million EUR'      THEN value_parsed
            WHEN 'percentage'       THEN value_parsed
            WHEN 'persons'          THEN value_parsed
            WHEN 'lost time per million hours of work' THEN value_parsed
            ELSE NULL
        END AS value,
        CASE
            WHEN unit_as_reported LIKE '%CO2e' THEN 't CO2e'
            WHEN unit_as_reported LIKE '%MWh'  THEN 'MWh'
            ELSE unit_as_reported
        END AS unit
    FROM typed
)

SELECT
    company,
    report_year,
    kpi_name,
    value,
    unit,
    scope_or_note,
    scope_or_note LIKE '%restated%' AS is_restated,
    value_as_reported,
    unit_as_reported,
    source_document,
    source_page,
    _ingested_at
FROM normalized;

In [0]:
SELECT 'value could not be parsed or unit unknown' AS test, COUNT(*) AS failing_rows
FROM workspace.silver.utility_kpis WHERE value IS NULL
UNION ALL
SELECT 'duplicate company / year / KPI', COUNT(*) FROM (
    SELECT company, report_year, kpi_name FROM workspace.silver.utility_kpis
    GROUP BY ALL HAVING COUNT(*) > 1)
UNION ALL
SELECT 'negative value', COUNT(*)
FROM workspace.silver.utility_kpis WHERE value < 0
UNION ALL
SELECT 'percentage outside 0-100', COUNT(*)
FROM workspace.silver.utility_kpis WHERE unit = 'percentage' AND value NOT BETWEEN 0 AND 100
UNION ALL
SELECT 'more female employees than employees', COUNT(*) FROM(
    SELECT company, report_year
    FROM workspace.silver.utility_kpis
    GROUP BY ALL
    HAVING MAX(CASE WHEN kpi_name = 'Female employees (heacount)' THEN value END)
        >  MAX(CASE WHEN kpi_name = 'Employees (headcount)' THEN value END));

In [0]:
SELECT kpi_name, report_year, company, value, unit, is_restated
FROM workspace.silver.utility_kpis
ORDER BY kpi_name, report_year, company;